# DAP-seq Report Notebook

Use this notebook to query the shared pipeline database, filter runs, and regenerate the HTML/TSV report for any subset of samples.

**Setup:** Copy this notebook to your own directory before editing — don't modify the template in the pipeline repo.

In [ ]:
import sqlite3
import sys
import pandas as pd
from pathlib import Path

# Path to the shared pipeline database
DB_PATH = "/path/to/pipeline/pipeline_db.db"

# Make report.py helpers importable
PIPELINE_DIR = "/path/to/pipeline"
sys.path.insert(0, str(Path(PIPELINE_DIR) / "workflow" / "scripts"))
from report import COLS, write_html, logo_to_base64

## Load the database

In [ ]:
con = sqlite3.connect(DB_PATH)
df      = pd.read_sql("SELECT * FROM pipeline_runs", con)
df_meta = pd.read_sql("SELECT * FROM run_metadata", con)
con.close()

print(f"{len(df)} rows, {df['output_dir'].nunique()} unique runs")
df.head()

## Explore

In [ ]:
# All unique runs in the database
df[["output_dir", "run_date", "genome_ref"]].drop_duplicates().sort_values("run_date", ascending=False)

In [ ]:
# Numeric summary of QC columns
numeric_cols = ["total_frags", "clean_reads", "filtered_reads", "peak#", "min5fold_peak#", "FRiP_score"]
df[numeric_cols].apply(pd.to_numeric, errors="coerce").describe()

### Metadata (authors and file paths)

In [ ]:
# Provenance summary: who ran what, and key output paths
_META_PREVIEW_COLS = [
    "output_dir", "sample", "author", "run_date",
    "genome_ref", "gene_annotation",
    "r1", "r2",
    "bam", "bigwig", "peaks_filt_narrowpeak",
    "meme_summits_dir", "report_html",
]
df_meta[[c for c in _META_PREVIEW_COLS if c in df_meta.columns]].sort_values(
    ["run_date", "sample"], ascending=[False, True]
)

## Filter

Edit the cell below to select the rows you want in the report.
Each example is independent — combine them with `&` as needed.

In [ ]:
filtered = df.copy()

# --- filter by specific output directory ---
# filtered = filtered[filtered["output_dir"] == "/scratch/myproject/run1"]

# --- filter by sample name (substring match) ---
# filtered = filtered[filtered["sample"].str.contains("TF1")]

# --- filter by date range ---
# filtered = filtered[filtered["run_date"] >= "2025-01-01"]
# filtered = filtered[filtered["run_date"].between("2025-01-01", "2025-06-30")]

# --- filter by genome reference ---
# filtered = filtered[filtered["genome_ref"].str.contains("GRCh38")]

# --- filter by minimum FRiP score ---
# filtered = filtered[pd.to_numeric(filtered["FRiP_score"], errors="coerce") >= 5.0]

# --- filter to treatment samples only (exclude controls) ---
# filtered = filtered[filtered["is_treatment"] == "True"]

# --- filter by author (cross-references run_metadata) ---
# filtered = filtered[df_meta.set_index(["output_dir", "sample"])
#                             .reindex(filtered.set_index(["output_dir", "sample"]).index)
#                             .reset_index()["author"] == "Nick"]

# Sync metadata to the same (output_dir, sample) pairs as filtered
filtered_meta = df_meta.merge(
    filtered[["output_dir", "sample"]].drop_duplicates(),
    on=["output_dir", "sample"],
)

print(f"{len(filtered)} rows selected  |  {len(filtered_meta)} metadata rows")
filtered[["sample", "output_dir", "run_date", "peak#", "FRiP_score"]]

## Generate report

`generate_report(df, out_dir)` writes `report.tsv` and `report.html` to the directory you specify.
It tries to load motif logos from the original output directories; logos are silently skipped if the path no longer exists.

In [ ]:
# Column name mapping: DB stores underscored names, report.py expects spaced names
_DB_TO_REPORT = {
    "min5fold_peak#": "min5fold peak#",
    "max_peak_score": "max peak score",
    "peak_reads#":    "peak reads#",
}

# Metadata columns shown in the HTML provenance section (condensed view)
_META_HTML_COLS = [
    "sample", "author", "run_date", "genome_ref", "gene_annotation",
    "r1", "r2", "bam", "bigwig",
    "peaks_filt_narrowpeak", "meme_summits_dir", "report_html",
]


def _build_meta_html(df_meta):
    """Return an HTML <details> block with a styled provenance table."""
    cols = [c for c in _META_HTML_COLS if c in df_meta.columns]
    header = "".join(f"<th style='padding:4px 8px;text-align:left'>{c}</th>" for c in cols)
    rows_html = ""
    for i, (_, row) in enumerate(df_meta.iterrows()):
        bg = "#f7f7f7" if i % 2 == 0 else "#ffffff"
        cells = "".join(
            f"<td style='padding:4px 8px;border-top:1px solid #e0e0e0'>{row.get(c, '')}</td>"
            for c in cols
        )
        rows_html += f"<tr style='background:{bg}'>{cells}</tr>\n"

    return f"""
<details style="margin-top:2em">
  <summary style="cursor:pointer;font-size:1.1em;font-weight:bold;
                  padding:0.4em 0.8em;background:#2c3e50;color:#fff;
                  border-radius:4px;list-style:none">
    &#9660; Data Provenance (file paths &amp; authorship)
  </summary>
  <div style="overflow-x:auto;margin-top:0.5em">
    <table style="border-collapse:collapse;font-size:0.82em;width:100%;
                  font-family:monospace">
      <thead>
        <tr style="background:#2c3e50;color:#fff">{header}</tr>
      </thead>
      <tbody>{rows_html}</tbody>
    </table>
  </div>
</details>
"""


def generate_report(df, out_dir, df_meta=None, include_metadata=True):
    """Write report.tsv and report.html for the given filtered DataFrame.

    If include_metadata is True and df_meta is provided, also writes
    report_metadata.tsv and appends a collapsible provenance section to
    the HTML report.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = df.rename(columns=_DB_TO_REPORT)
    rows = df.to_dict(orient="records")

    # Try to load motif logos from the original output directories
    logo_b64_map = {}
    for row in rows:
        sample     = row["sample"]
        output_dir = row["output_dir"]
        logo_path  = Path(output_dir) / "meme" / sample / "summits" / "logo1.png"
        logo_b64_map[sample] = logo_to_base64(str(logo_path))

    # Write TSV (QC metrics)
    tsv_path = out_dir / "report.tsv"
    with open(tsv_path, "w") as fh:
        fh.write("\t".join(COLS) + "\n")
        for row in rows:
            fh.write("\t".join(str(row.get(c, "NA")) for c in COLS) + "\n")

    # Write HTML (QC report)
    html_path = out_dir / "report.html"
    write_html(rows, logo_b64_map, str(html_path))

    # Optionally append collapsible metadata provenance section to HTML
    if include_metadata and df_meta is not None and len(df_meta) > 0:
        meta_section = _build_meta_html(df_meta)
        html_content = html_path.read_text()
        html_content = html_content.replace("</body>", meta_section + "\n</body>")
        html_path.write_text(html_content)

        meta_tsv_path = out_dir / "report_metadata.tsv"
        df_meta.to_csv(meta_tsv_path, sep="\t", index=False)
        print(f"Written: {meta_tsv_path}")

    print(f"Written: {tsv_path}")
    print(f"Written: {html_path}")

In [ ]:
# Set to True to append a "Data Provenance" section to the HTML report
# and write report_metadata.tsv alongside the QC report.
INCLUDE_METADATA = True

# Edit the output directory to wherever you want the report files
generate_report(
    filtered,
    "/path/to/my/report_output",
    df_meta=filtered_meta if INCLUDE_METADATA else None,
    include_metadata=INCLUDE_METADATA,
)